<a href="https://colab.research.google.com/github/rohitblpprajapat/100-days-of-code/blob/master/continual_pretraining_of_llama_3_2_1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pprint import pprint
import math
import wandb

import datasets
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import DataCollatorForLanguageModeling
from transformers import TrainingArguments, Trainer
from huggingface_hub import login

In [ ]:
wandb.init(
    project="DLP-w4-cpt-node-1",
    config={
        "batch_size":4,
        "dataset":"Sangraha"

    }
)

In [3]:
login()

In [4]:
ds = load_dataset("ai4bharat/sangraha", data_files="https://huggingface.co/datasets/ai4bharat/sangraha/resolve/main/verified/tam/data-0.parquet")

README.md: 0.00B [00:00, ?B/s]

verified/tam/data-0.parquet:   0%|          | 0.00/358M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'text', 'type'],
        num_rows: 149796
    })
})

In [6]:
model_id = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Context length: {tokenizer.model_max_length}')

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Vocab size: 128000
Context length: 131072


In [7]:
tokenizer.model_max_length = 1024
tokenizer.pad_token = tokenizer.eos_token

In [8]:
eg = ds['train'][1]
num_words = len(eg['text'].split())
print(num_words)

47


In [9]:
input_ids = tokenizer(eg['text'])['input_ids']
print(input_ids)
print(len(input_ids))

[128000, 20627, 248, 32601, 228, 20627, 107, 64500, 106, 84298, 20627, 109, 32601, 230, 20627, 225, 198, 20627, 103, 20627, 248, 64500, 248, 20627, 108, 100112, 248, 91702, 71697, 106, 20627, 109, 64500, 109, 84298, 20627, 106, 47454, 71697, 103, 20627, 248, 64500, 248, 32601, 230, 20627, 103, 64500, 103, 20627, 107, 20627, 109, 32601, 230, 71697, 240, 20627, 102, 64500, 109, 20627, 122, 20627, 243, 71697, 248, 32601, 229, 20627, 108, 64500, 97, 64500, 97, 84298, 71697, 240, 20627, 108, 84298, 71697, 106, 20627, 96, 91702, 71697, 101, 32601, 229, 20627, 108, 20627, 106, 47454, 71697, 232, 20627, 109, 71697, 113, 32601, 230, 20627, 243, 64500, 243, 20627, 113, 84298, 20627, 106, 47454, 13, 71697, 232, 20627, 109, 100112, 107, 71697, 227, 20627, 108, 100112, 248, 91702, 11, 71697, 103, 20627, 107, 20627, 109, 84298, 20627, 253, 20627, 102, 47454, 11, 71697, 97, 32601, 229, 20627, 247, 64500, 243, 20627, 122, 20627, 107, 47454, 71697, 97, 84298, 20627, 108, 84298, 20627, 113, 20627, 110, 

In [10]:
print(f'The fertility rate is: {len(input_ids)/num_words}')

The fertility rate is: 11.085106382978724


In [11]:
model = AutoModelForCausalLM.from_pretrained(model_id, pad_token_id =tokenizer.eos_token_id )

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [12]:
configuration = model.config
print(configuration)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128001,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 128256
}



In [13]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128001)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [14]:
num_parameters = 0
for param in model.parameters():
  num_parameters += param.numel()
print(f'Number of parameters : {num_parameters/10**9:.2f}B')

Number of parameters : 1.24B


In [15]:
print(model.dtype)

torch.bfloat16


In [16]:
mem_in_gb = num_parameters*2/1e9
mem_in_gb

2.4716288

In [17]:
model.get_memory_footprint()/1e9

2.471629056

In [19]:
prompt = "I was reading Geyman's lecture on physics. He talks about "
inputs = tokenizer(prompt, return_tensors='pt', padding=True)
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, top_k=10, top_p=0.95)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


["I was reading Geyman's lecture on physics. He talks about 3 things in his lecture:\n1) The 4th dimension is not a real dimension.\n2) The 4th dimension is a mathematical construct that is not real.\n3) The 4th dimension is a mathematical construct that is real.\n"]

In [20]:
print(inputs)
print(outputs)

{'input_ids': tensor([[128000,     40,    574,   5403,    480,   1216,   1543,    596,  31678,
            389,  22027,     13,   1283,  13739,    922,    220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
tensor([[128000,     40,    574,   5403,    480,   1216,   1543,    596,  31678,
            389,  22027,     13,   1283,  13739,    922,    220,     18,   2574,
            304,    813,  31678,    512,     16,      8,    578,    220,     19,
            339,  13167,    374,    539,    264,   1972,  13167,    627,     17,
              8,    578,    220,     19,    339,  13167,    374,    264,  37072,
           9429,    430,    374,    539,   1972,    627,     18,      8,    578,
            220,     19,    339,  13167,    374,    264,  37072,   9429,    430,
            374,   1972,    627]])


In [21]:
from peft import LoraConfig, TaskType, LoraModel
lora_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
    inference_mode = False,
    lora_alpha=32,
    lora_dropout=0.05
)

In [22]:
from peft import get_peft_model
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [23]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128001)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [24]:
training_args = TrainingArguments( output_dir = 'lora_llama_1b_ct',
                                  eval_strategy="steps",
                                   eval_steps=100,
                                   num_train_epochs=1,
                                   per_device_train_batch_size=2,
                                   per_device_eval_batch_size=2,
                                   bf16=False,
                                   fp16=True,
                                   tf32=False,
                                   gradient_accumulation_steps=1,
                                   adam_beta1=0.9,
                                   adam_beta2=0.999,
                                   learning_rate=2e-5,
                                   weight_decay=0.01,
                                   logging_dir='logs',
                                   logging_strategy='steps',
                                   logging_steps=100,
                                   save_steps=100,
                                   save_total_limit=20,
                                   report_to='none'
                                   )

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
trainer = Trainer(model=lora_model,
                  args=training_args,
                  train_dataset=ds_split["train"],
                  eval_dataset=ds_split["test"],
                  data_collator=data_collator
                  )

NameError: name 'ds_split' is not defined